# Phase 6: run the pre-registration CONSENT load

Writes ONE `ContactPointConsent` per staged row on purpose **invest_central**
(`0ZWTe0000000X5dOAE`), status from the batch suffix — the four batches from 03 all
qualify (`_optin` → OptIn with `consent_invest=1`, `_optout` → OptOut with
`consent_invest=0`; the loader enforces the pairing).

Prerequisites (04, all green): account loads verified, mirror refreshed, CPE backfill
done. Rows still without `sf_cp_email_id` are silently NOT loadable — section 1 counts
them; they stay a documented loose end (we never create CPEs).

Safety model as in the August run: dry-run → single-record probe with business
sign-off → gated bulk load → live verify. The loader skips CPEs that already carry an
invest_central consent and exits non-zero on status conflicts.

In [ ]:
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root
sys.path.insert(0, str(Path.cwd()))          # this job's loaders

from config import load_mysql_config
from mysql_client import MySQLClient

REPO_ROOT = Path.cwd().parent
BATCH_DATE = "2026-08-28"
CONSENT_BATCHES = [
    f"{BATCH_DATE}_prereg_update_optin",
    f"{BATCH_DATE}_prereg_update_optout",
    f"{BATCH_DATE}_prereg_insert_optin",
    f"{BATCH_DATE}_prereg_insert_optout",
]
DATA_USE_PURPOSE_ID = "0ZWTe0000000X5dOAE"   # invest_central

# August contract on this purpose (2026-08-25 run) - the verify in section 6
# expects these PLUS what this run writes.
AUGUST_OPTIN, AUGUST_OPTOUT = 6212, 10

# Loadable counts per batch - section 1 output, 2026-08-28 after the account phase.
EXPECTED = {
    f"{BATCH_DATE}_prereg_update_optin": 1696,
    f"{BATCH_DATE}_prereg_update_optout": 0,
    f"{BATCH_DATE}_prereg_insert_optin": 1495,
    f"{BATCH_DATE}_prereg_insert_optout": 0,
}

db = MySQLClient(load_mysql_config())
print("connected")

## 1. Reload with the loader's own predicate

Per batch: loadable rows (CPE present, flag matches suffix, consent still open) and
the not-loadable leftovers (no CPE). Fill `EXPECTED` from this output, re-run, and
only then continue.

In [ ]:
for b in CONSENT_BATCHES:
    flag = 1 if b.endswith("_optin") else 0
    r = db.fetch_one("""
        SELECT SUM(_excluded = 0 AND sf_cp_email_id IS NOT NULL
                   AND consent_invest = %s AND _consent_processed_at IS NULL) AS loadable,
               SUM(_excluded = 0 AND sf_cp_email_id IS NULL) AS no_cpe,
               SUM(_excluded = 0 AND consent_invest <> %s) AS wrong_flag
        FROM crm_imp_person_accounts WHERE _batch_id = %s
    """, (flag, flag, b))
    print(f"{b}: loadable {int(r['loadable'] or 0):,} | no CPE {int(r['no_cpe'] or 0)} "
          f"| wrong flag {int(r['wrong_flag'] or 0)}")
    assert int(r["wrong_flag"] or 0) == 0, "a row's consent_invest contradicts its batch suffix"
    if EXPECTED[b] is not None:
        assert int(r["loadable"] or 0) == EXPECTED[b], f"drift on {b}"

assert all(v is not None for v in EXPECTED.values()), "fill EXPECTED from the counts above, then re-run"
print("reload contract holds")

## 2. Payload preview + dry-runs

Five payloads through the real `row_to_sf_record`, then `--dry-run` per batch
(CSV to `local_data/dry_run_prereg_consents_*.csv`, no Salesforce contact).

In [ ]:
from datetime import datetime, timezone
from insert_consents_prereg import row_to_sf_record, sf_datetime

now = sf_datetime(datetime.now(timezone.utc))
sample = db.fetch_all(
    "SELECT * FROM crm_imp_person_accounts WHERE _batch_id = %s AND _excluded = 0 "
    "AND sf_cp_email_id IS NOT NULL LIMIT 5",
    (CONSENT_BATCHES[0],),
)
for row in sample:
    print(row_to_sf_record(row, "OptIn", now))

def run_loader(batch_id: str, dry_run: bool = False) -> None:
    cmd = [sys.executable, str(Path.cwd() / "insert_consents_prereg.py"), batch_id] \
          + (["--dry-run"] if dry_run else [])
    print(">>", " ".join(cmd[1:]))
    res = subprocess.run(cmd, cwd=REPO_ROOT)
    assert res.returncode == 0, f"insert_consents_prereg.py {batch_id} exited {res.returncode}"

for b in CONSENT_BATCHES:
    run_loader(b, dry_run=True)

## 3. Live pre-check: existing consents on this purpose

Read-only SOQL over all staged CPE ids. Expected hits: the ~2 accounts that were
already invest-flagged (recon query 9) plus anything from the August run that
overlaps. The loader will skip exactly these — a big number here means the
populations overlap more than recon predicted: stop and investigate.

In [ ]:
from salesforce_client_prod import SalesforceClientCC, load_salesforce_cc_config_from_env
from insert_consents_prereg import existing_consents

cp_ids = [r["sf_cp_email_id"] for r in db.fetch_all("""
    SELECT DISTINCT sf_cp_email_id FROM crm_imp_person_accounts
    WHERE _batch_id IN (%s, %s, %s, %s) AND _excluded = 0 AND sf_cp_email_id IS NOT NULL
""", tuple(CONSENT_BATCHES))]
print(f"{len(cp_ids):,} distinct CPE ids staged")

with SalesforceClientCC(load_salesforce_cc_config_from_env()) as sf:
    sf.authenticate()
    found = existing_consents(sf, cp_ids)
print(f"CPEs with an existing invest_central consent: {len(found)}")
for cp, recs in list(found.items())[:10]:
    print(" ", cp, [(r["PrivacyConsentStatus"], r["CaptureDate"]) for r in recs])

## 4. Probe: ONE consent insert

Business-agreed account from the optin update batch, hand-entered. Single REST POST,
then readback of the consent, the CPE it hangs on (proving it is the right person),
and the consent count on that CPE. The bulk load skips it afterwards via the
existing-consent check.

In [ ]:
RUN_PROBE = False
PROBE_ACCOUNT_ID = ""   # hand-entered, business-agreed

if RUN_PROBE:
    from datetime import datetime, timezone
    from insert_consents_prereg import row_to_sf_record, sf_datetime

    row = db.fetch_one("""
        SELECT * FROM crm_imp_person_accounts
        WHERE _batch_id = %s AND sf_account_id = %s AND _excluded = 0
          AND sf_cp_email_id IS NOT NULL
    """, (CONSENT_BATCHES[0], PROBE_ACCOUNT_ID))
    assert row, "probe account not loadable in the optin update batch"
    payload = row_to_sf_record(row, "OptIn", sf_datetime(datetime.now(timezone.utc)))
    print("payload:", payload)

    with SalesforceClientCC(load_salesforce_cc_config_from_env()) as sf:
        sf.authenticate()
        r = sf._client.post(f"{sf._base()}/sobjects/ContactPointConsent/", json=payload)
        assert r.status_code == 201, f"probe insert failed ({r.status_code}): {r.text}"
        consent_id = r.json()["id"]
        print("created:", consent_id)

        back = sf.query(
            "SELECT Id, Name, ContactPointId, DataUsePurposeId, PrivacyConsentStatus, "
            "CaptureDate, EffectiveFrom, ConsentKey__c, CaptureSource, SourceSystem__c "
            f"FROM ContactPointConsent WHERE Id = '{consent_id}'"
        )["records"][0]
        back.pop("attributes", None)
        for k, v in back.items():
            print(f"  {k:22s} {v}")

        cpe = sf.query(
            "SELECT Id, ParentId, EmailAddress "
            f"FROM ContactPointEmail WHERE Id = '{row['sf_cp_email_id']}'"
        )["records"][0]
        print("  CPE:", cpe["Id"], cpe["EmailAddress"], "| staged email:", row["email"])

        cnt = sf.query(
            "SELECT COUNT(Id) c FROM ContactPointConsent "
            f"WHERE ContactPointId = '{row['sf_cp_email_id']}'"
        )["records"][0]["c"]
        print("  consents on this CPE:", cnt)
else:
    print("probe gated (RUN_PROBE = False)")

## 5. Bulk load

**Gated. Only after explicit go-ahead.** One loader call per batch; the suffix picks
the status, the loader re-checks existing consents, aborts on status conflicts, and
writes `_consent_processed_at` per Bulk batch.

In [ ]:
RUN_LOAD = False

if RUN_LOAD:
    for b in CONSENT_BATCHES:
        run_loader(b)
else:
    print("load gated (RUN_LOAD = False)")

## 6. Verify

Live count by status on the purpose vs August contract + this run; per-batch
`still_open` (with a CPE) must be 0; no CPE may carry more than one invest_central
consent.

In [ ]:
from salesforce_client_prod import SalesforceClientCC, load_salesforce_cc_config_from_env

with SalesforceClientCC(load_salesforce_cc_config_from_env()) as sf:
    sf.authenticate()
    counts = {r["PrivacyConsentStatus"]: r["expr0"] for r in sf.query_all(
        "SELECT PrivacyConsentStatus, COUNT(Id) FROM ContactPointConsent "
        f"WHERE DataUsePurposeId = '{DATA_USE_PURPOSE_ID}' GROUP BY PrivacyConsentStatus"
    )["records"]}
    print("live invest_central:", counts)

    new_optin  = sum(EXPECTED[b] for b in CONSENT_BATCHES if b.endswith("_optin"))
    new_optout = sum(EXPECTED[b] for b in CONSENT_BATCHES if b.endswith("_optout"))
    print(f"expected: OptIn ~{AUGUST_OPTIN + new_optin:,} | OptOut ~{AUGUST_OPTOUT + new_optout:,} "
          "(minus loader skips - reconcile against the skipped_* files)")

    dupes = sf.query_all(
        "SELECT ContactPointId, COUNT(Id) FROM ContactPointConsent "
        f"WHERE DataUsePurposeId = '{DATA_USE_PURPOSE_ID}' "
        "GROUP BY ContactPointId HAVING COUNT(Id) > 1"
    )["records"]
    print(f"CPEs with >1 invest_central consent: {len(dupes)}")
    assert not dupes

for b in CONSENT_BATCHES:
    flag = 1 if b.endswith("_optin") else 0
    open_ = db.fetch_one("""
        SELECT SUM(_excluded = 0 AND sf_cp_email_id IS NOT NULL
                   AND consent_invest = %s AND _consent_processed_at IS NULL) AS still_open
        FROM crm_imp_person_accounts WHERE _batch_id = %s
    """, (flag, b))["still_open"]
    print(f"{b}: still open {int(open_ or 0)}")
    # Loader-skipped rows (pre-existing consent) keep _consent_processed_at NULL by
    # design - reconcile them against local_data/skipped_prereg_consents_*.json
    # before calling this 0-or-explained.

## 7. Archive

**Gated.** Snapshots each batch into `crm_imp_person_accounts_history` and deletes it
from the live staging table (`sp_archive_crm_imp_person_accounts` rolls back on count
mismatch). Only after section 6 is signed off and the loose ends (no-CPE rows,
loader skips) are documented.

In [ ]:
ARCHIVE = False   # as-run 2026-08-28: both non-empty batches archived (see note below)

# sp_archive_crm_imp_person_accounts returns a summary result set -
# db.execute() does not consume it and dies with "Unread result found"
# AFTER the proc's work is committed. Archive therefore via raw connector.
# As-run: update_optin (1,696) archived by the first (crashing) db.execute call,
# insert_optin (1,496 incl. the Daurer exclude) via this cell; the _optout
# batches hold 0 rows - nothing to archive. History total 3,192, live 0.
if ARCHIVE:
    import mysql.connector
    from config import load_mysql_config
    cfg = load_mysql_config()
    conn = mysql.connector.connect(**(cfg if isinstance(cfg, dict) else vars(cfg)))
    cur = conn.cursor(dictionary=True)
    for b in CONSENT_BATCHES:
        n = db.fetch_one(
            "SELECT COUNT(*) AS n FROM crm_imp_person_accounts WHERE _batch_id = %s", (b,)
        )["n"]
        if not n:
            print("skip (empty batch):", b)
            continue
        cur.execute("CALL sp_archive_crm_imp_person_accounts(%s, %s)", (b, "arsal_prereg_2026-08"))
        while True:
            if cur.with_rows:
                for r in cur.fetchall():
                    print(" ", r)
            if not cur.nextset():
                break
        conn.commit()
        print("archived", b)
    cur.close(); conn.close()
else:
    print("archive gated (ARCHIVE = False)")